In [1]:
# Guía de Pandas para el ETL de GAMECON
# Generado para: Andrés Hormazábal — UDP
# Material de apoyo del sistema de 25 notebooks (acompaña a NB-02)
# Fecha de generación: 2026-10-30

# Guía de Pandas para el ETL

**Bloque:** 1 — Fundamentos de Datos y Cómputo · *material de apoyo de NB-02*

**Base bibliográfica:** VanderPlas, *Python Data Science Handbook* (cap. 13–24:
objetos de pandas, indexación, agregación, pivot, operaciones de texto) ·
McKinney, *Python for Data Analysis* (cap. 5–8) como catálogo de referencia

**Prerequisitos:** NB-01 (NumPy, arrays, broadcasting). Python básico.

## Para qué existe esta guía

NB-02 enseña el **flujo completo** pandas → MongoDB sobre un caso de ejemplo. Esta
guía es lo otro: el **catálogo de referencia**, ordenado según el flujo real del ETL
—extraer → perfilar → limpiar → transformar → agrupar → cargar— para que cuando
estés programando sepas exactamente qué función usar en qué paso.

> **Para tu grupo.** Esta guía está escrita para que Cris y Sofi también puedan usarla
> sin haber leído NB-02 completo. Cada sección termina con un bloque
> **🎯 Para tu proyecto** que dice **qué aplicar**, sin resolverlo: el diseño del ETL
> de GAMECON es decisión del grupo y es trabajo evaluado.

## Objetivos de Aprendizaje

Al completar esta guía serás capaz de:

1. **Elegir** la función de carga correcta según formato y tamaño del archivo.
2. **Perfilar** cualquier dataset desconocido en menos de diez líneas y saber qué
   mirar primero.
3. **Normalizar** texto y etiquetas de forma reproducible, que es el paso que decide
   si Jaccard sirve o no.
4. **Convertir** una columna de "varios valores en un string" a forma larga y de vuelta.
5. **Construir** una tabla de stands agregando por productor, sin perder etiquetas.
6. **Preparar** documentos BSON-válidos para MongoDB desde un DataFrame.

| Curso / Área | Relevancia |
|---|---|
| **Big Data (CIT-2309)** | **Referencia directa de la Entrega 2**: el ETL del proyecto usa exactamente estas funciones. |
| IA (CIT-2013) | Preparar datos es el 80 % del trabajo antes de entrenar cualquier modelo (NB-09, NB-11). |
| Comunicaciones Digitales | Series temporales de mediciones, `.dt`, ventanas móviles. |
| Sistemas Distribuidos | El paso de "DataFrame en memoria" a "documentos distribuidos" empieza en §12. |
| Tesis SNN/Ojo | Registro de experimentos (parámetros + métricas) como tabla y como documento. |
| Notebooks relacionados | NB-01 (base), **NB-02** (el flujo completo), NB-03 (vectorización) |

<a id="indice"></a>
## Índice

Haz clic en una sección para saltar a ella; cada sección tiene un enlace *↑ índice* para volver aquí.

- [Configuración](#config)
- [0. Datos de juguete](#sec-0) · [1. Cargar datos](#sec-1) · [2. Perfilar](#sec-2) · [3. Seleccionar y filtrar](#sec-3) · [4. Limpieza y tipos](#sec-4) · [5. Texto con .str](#sec-5) · [6. Columnas con varios valores](#sec-6) · [7. Vocabulario común y tabla de mapeo](#sec-7) · [8. Agrupar y agregar](#sec-8) · [9. Unir tablas](#sec-9) · [10. Frecuencias, IDF y similitud](#sec-10) · [11. Datos sintéticos](#sec-11) · [12. Salida](#sec-12) · [13. Cheat sheet y mapa del proyecto](#sec-13) · [14. Conexiones con el mundo real](#sec-14) · [15. Ejercicios progresivos](#sec-15) · [16. Glosario de errores frecuentes](#sec-16) · [17. Resumen y siguiente paso](#sec-17)


<a id="config"></a>
## Configuración

Una nota sobre cómo está escrita esta guía, porque cambia cómo se lee.

La versión anterior apilaba diez expresiones en una celda, así:

```python
toy.shape
toy.dtypes
toy.isna().sum()      # ← SOLO ESTA se ve
```

Jupyter muestra únicamente **la última** expresión de cada celda, así que nueve de
cada diez líneas quedaban sin salida visible. Aquí se arregla con un helper `ver()`
que imprime una etiqueta y el resultado, de modo que **cada línea muestre lo suyo**.
Es un detalle pequeño con un efecto grande: puedes leer la guía sin ejecutarla.

In [32]:
import ast                      # para convertir texto que "parece lista" en lista real
import unicodedata              # para quitar tildes de forma correcta (no con replace)
from io import StringIO         # para simular archivos en memoria, sin tocar el disco
from pathlib import Path        # rutas que funcionan igual en macOS, Linux y Windows

import numpy as np              # arrays y números aleatorios
import pandas as pd             # el protagonista de esta guía

from IPython.display import display   # para mostrar DataFrames con formato dentro de ver()

print("pandas", pd.__version__, "| numpy", np.__version__)

# --- Opciones de visualización: afectan solo a cómo se IMPRIME, no a los datos ---
pd.set_option("display.max_columns", 30)    # no cortar en 10 columnas
pd.set_option("display.max_colwidth", 60)   # ancho máximo de una celda de texto
pd.set_option("display.width", 160)         # ancho total de la consola antes de partir líneas


def ver(etiqueta: str, obj, maxfilas: int = 8) -> None:
    """Muestra `obj` con una etiqueta, sea lo que sea.

    Existe por una razón concreta: en una celda con varias expresiones, Jupyter
    solo muestra la ÚLTIMA. Con ver() cada línea deja su rastro, y la guía se
    puede leer sin ejecutarla.
    """
    print(f"\n▸ {etiqueta}")                         # separador visual con la etiqueta
    if isinstance(obj, (pd.DataFrame, pd.Series)):   # si es tabla o serie...
        display(obj.head(maxfilas) if len(obj) > maxfilas else obj)   # ...recorta y formatea
    else:                                            # cualquier otra cosa (int, dict, array)
        print(obj)

pandas 3.0.6 | numpy 2.5.3


---

<a id="sec-0"></a>
## 0. Datos de juguete
<sub>[↑ índice](#indice)</sub>

Estos datos **imitan la forma** de tus tres fuentes (vgsales, anime, Goodreads), no su
contenido: un productor, géneros como texto separado por comas, nulos, duplicados y
tipos mezclados.

Usar datos de juguete no es pereza: es que **conoces la respuesta**. Si `drop_duplicates`
debe eliminar exactamente una fila, puedes verificarlo. Con 16.598 filas de vgsales no
puedes.

In [33]:
toy = pd.DataFrame({
    # Nombre del título. El None del final sirve para practicar con nulos en texto.
    "Name":      ["Alpha", "Beta", "Gamma", "Gamma", "Delta", "Epsilon", None],
    # Productor. Ojo a propósito: " Acme " y "acme" son el MISMO productor escrito
    # distinto (espacios + mayúsculas). Ese es el problema central de §5.
    "Publisher": [" Acme ", "acme", "Zeta Co", "Zeta Co", "Omega", None, "Omega"],
    # Año MEZCLADO: unos int, otros str, un NaN -> pandas lo deja como 'object'.
    "Year":      [2001, 2005, "2010", "2010", np.nan, 1999, 2020],
    # Géneros: varios valores dentro de UN string. Es la forma de vgsales y anime.
    "Genres":    ["Action, Fantasy", "Action", "Sci-Fi, Drama, Action",
                  "Sci-Fi, Drama, Action", None, "Fantasy", "Drama"],
    # Una columna numérica limpia, para los ejemplos de filtrado y agregación.
    "Sales":     [1.5, 0.3, 9.9, 9.9, 0.1, 2.2, 0.7],
})

ver("El DataFrame completo", toy)
ver("Defectos inyectados a propósito", {
    "filas duplicadas exactas": int(toy.duplicated().sum()),
    "nulos por columna": toy.isna().sum().to_dict(),
    "dtype de 'Year' (mezcla int/str/NaN)": str(toy["Year"].dtype),
})


▸ El DataFrame completo


,Name,Publisher,Year,Genres,Sales
0,Alpha,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
4,Delta,Omega,NaN,NaN,0.1
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,Omega,2020,Drama,0.7



▸ Defectos inyectados a propósito
{'filas duplicadas exactas': 1, 'nulos por columna': {'Name': 1, 'Publisher': 1, 'Year': 1, 'Genres': 1, 'Sales': 0}, "dtype de 'Year' (mezcla int/str/NaN)": 'object'}


---

<a id="sec-1"></a>
## 1. Cargar datos
<sub>[↑ índice](#indice)</sub>

| Función | Para qué |
|---|---|
| `pd.read_csv` | CSV, también `.csv.zip` / `.gz` (la compresión se infiere del nombre) |
| `pd.read_json(lines=True)` | JSON Lines — **el formato de Goodreads** |
| `pd.read_parquet` | formato columnar; conserva tipos, mucho más rápido |
| `read_csv(chunksize=)` / `read_json(chunksize=)` | archivos que no caben en memoria |

> **NOTA.** El orden correcto al abrir un archivo desconocido es: primero `nrows=1000`
> para mirar, **después** la carga completa. Cargar 89.000 registros JSON anidados para
> descubrir que la columna que buscabas se llama distinto es tiempo perdido.

In [34]:
RAIZ = Path.cwd()                       # carpeta desde donde se ejecuta el notebook
RAW = RAIZ / "datos" / "raw"            # convención del proyecto: datos crudos aquí
print("Buscaría los datos crudos en:", RAW)

# --- Los parámetros de read_csv que de verdad vas a usar ---
# Se muestran como texto porque dependen de TUS archivos; abajo hay una demo ejecutable.
ayuda_read_csv = """
pd.read_csv(RAW / "vgsales.csv.zip",
    usecols=["Name", "Publisher", "Genre"],  # leer SOLO estas columnas -> menos memoria
    dtype={"Publisher": "string"},           # forzar tipo (evita que infiera object)
    nrows=1000,                              # solo las primeras N filas, para explorar
    encoding="utf-8",                        # "latin-1" si salen caracteres raros
    sep=",",                                 # separador; ";" en CSV europeos
    na_values=["N/A", "-", "", "Unknown"],   # qué TEXTO debe contar como nulo
    parse_dates=["fecha"],                   # convertir a datetime durante la lectura
)
"""
print(ayuda_read_csv)

Buscaría los datos crudos en: /Users/andres/Desktop/gamecon/datos/raw

pd.read_csv(RAW / "vgsales.csv.zip",
    usecols=["Name", "Publisher", "Genre"],  # leer SOLO estas columnas -> menos memoria
    dtype={"Publisher": "string"},           # forzar tipo (evita que infiera object)
    nrows=1000,                              # solo las primeras N filas, para explorar
    encoding="utf-8",                        # "latin-1" si salen caracteres raros
    sep=",",                                 # separador; ";" en CSV europeos
    na_values=["N/A", "-", "", "Unknown"],   # qué TEXTO debe contar como nulo
    parse_dates=["fecha"],                   # convertir a datetime durante la lectura
)



In [35]:
# --- Demo ejecutable: un CSV que vive en memoria, sin tocar el disco ---
csv_demo = StringIO("a,b\n1,x\n2,N/A\n3,z")        # StringIO finge ser un archivo abierto

sin_na = pd.read_csv(StringIO("a,b\n1,x\n2,N/A\n3,z"))          # sin declarar na_values
con_na = pd.read_csv(csv_demo, na_values=["N/A"])               # declarando que "N/A" es nulo

ver("Sin na_values: 'N/A' queda como TEXTO", sin_na)
ver("dtype de la columna b (sin na_values)", sin_na["b"].dtype)
ver("Con na_values=['N/A']: pasa a ser NaN", con_na)
ver("¿Cuántos nulos detecta ahora?", con_na.isna().sum().to_dict())
print("""
  Esta diferencia importa más de lo que parece: con 'N/A' como texto, la columna
  es de tipo texto y cualquier operación numérica falla. Declarar na_values en la
  LECTURA es más barato que arreglarlo después.""")


▸ Sin na_values: 'N/A' queda como TEXTO


,a,b
0,1,x
1,2,NaN
2,3,z



▸ dtype de la columna b (sin na_values)
str

▸ Con na_values=['N/A']: pasa a ser NaN


,a,b
0,1,x
1,2,NaN
2,3,z



▸ ¿Cuántos nulos detecta ahora?
{'a': 0, 'b': 1}

  Esta diferencia importa más de lo que parece: con 'N/A' como texto, la columna
  es de tipo texto y cualquier operación numérica falla. Declarar na_values en la
  LECTURA es más barato que arreglarlo después.


In [36]:
# --- Archivos que no caben en memoria: leer por pedazos ---
# El patrón es siempre el mismo: procesar cada trozo y guardar SOLO el resumen,
# nunca los trozos completos (si los acumulas, volviste al problema original).
fuente = StringIO("\n".join(["v"] + [str(i) for i in range(1000)]))   # CSV de 1000 filas

suma, n = 0.0, 0                                    # acumuladores: dos números, no una tabla
for trozo in pd.read_csv(fuente, chunksize=250):    # chunksize -> devuelve un ITERADOR
    suma += trozo["v"].sum()                        # agregamos lo que necesitamos del trozo
    n += len(trozo)                                 # y contamos filas
    # al terminar la vuelta del for, `trozo` se descarta: la memoria no crece

ver("Filas procesadas por pedazos", n)
ver("Media calculada sin cargar el archivo entero", suma / n)
ver("Comprobación con la fórmula de Gauss", sum(range(1000)) / 1000)

print("""
🎯 Para tu proyecto
   · vgsales y anime son CSV chicos: cárgalos completos.
   · Goodreads son 89.411 registros JSON anidados -> pd.read_json(..., lines=True)
     y durante la EXPLORACIÓN usa nrows=1000. El archivo completo solo cuando ya
     sepas qué columnas quieres.
   · Pregunta de diseño que tienes que responder: ¿qué columnas necesitas de cada
     fuente? Pasar usecols= es la diferencia entre 200 MB y 20 MB en RAM.""")


▸ Filas procesadas por pedazos
1000

▸ Media calculada sin cargar el archivo entero
499.5

▸ Comprobación con la fórmula de Gauss
499.5

🎯 Para tu proyecto
   · vgsales y anime son CSV chicos: cárgalos completos.
   · Goodreads son 89.411 registros JSON anidados -> pd.read_json(..., lines=True)
     y durante la EXPLORACIÓN usa nrows=1000. El archivo completo solo cuando ya
     sepas qué columnas quieres.
   · Pregunta de diseño que tienes que responder: ¿qué columnas necesitas de cada
     fuente? Pasar usecols= es la diferencia entre 200 MB y 20 MB en RAM.


---

<a id="sec-2"></a>
## 2. Perfilar: conocer el dataset
<sub>[↑ índice](#indice)</sub>

**Perfilar es lo primero que se hace con un archivo que no escribiste tú.** Y hay un
orden que conviene respetar, porque cada pregunta condiciona la siguiente:

```
1. ¿Qué forma tiene?        shape, columns, dtypes
2. ¿Qué falta?              isna().sum(), isna().mean()
3. ¿Qué se repite?          duplicated(), nunique()
4. ¿Cómo se distribuye?     describe(), value_counts()
5. ¿Cuánto ocupa?           memory_usage(deep=True)
```

> ⚠️ **La medición que decide si Jaccard te sirve.** Para la columna de géneros, lo
> importante no es cuántos valores distintos hay, sino **cuántos valores trae cada
> fila**. Si la mediana es 1, Jaccard degenera en "coinciden o no coinciden" y tu
> sistema de similitud deja de tener sentido. Esa medición está en §6.

In [37]:
ver("1 · Forma (filas, columnas)", toy.shape)
ver("1 · Nombres de columna", toy.columns.tolist())
ver("1 · Tipo de cada columna", toy.dtypes)

ver("2 · Nulos por columna (cuenta)", toy.isna().sum())
ver("2 · Nulos por columna (proporción)", toy.isna().mean().round(3))

ver("3 · Filas duplicadas EXACTAS", int(toy.duplicated().sum()))
ver("3 · Duplicados según una sola columna", int(toy.duplicated(subset=["Name"]).sum()))
ver("3 · Valores distintos por columna", toy.nunique())

ver("4 · Estadísticas de columnas numéricas", toy.describe())
ver("4 · Frecuencia de cada productor", toy["Publisher"].value_counts())
ver("4 · ...contando también los nulos", toy["Publisher"].value_counts(dropna=False))
ver("4 · ...como proporciones", toy["Publisher"].value_counts(normalize=True).round(3))

ver("5 · Memoria por columna (bytes)", toy.memory_usage(deep=True))
print("""
  Dos detalles que se pasan por alto:

  · value_counts() DESCARTA los nulos por defecto. Si quieres saber cuántos
    registros no tienen productor, necesitas dropna=False. Es una fuente
    clásica de "me faltan filas y no sé dónde".
  · memory_usage necesita deep=True para medir el texto de verdad; sin él solo
    cuenta los punteros y te subestima el tamaño real.""")


▸ 1 · Forma (filas, columnas)
(7, 5)

▸ 1 · Nombres de columna
['Name', 'Publisher', 'Year', 'Genres', 'Sales']

▸ 1 · Tipo de cada columna


Name             str
Publisher        str
Year          object
Genres           str
Sales        float64
dtype: object


▸ 2 · Nulos por columna (cuenta)


Name         1
Publisher    1
Year         1
Genres       1
Sales        0
dtype: int64


▸ 2 · Nulos por columna (proporción)


...


▸ 4 · ...como proporciones


Publisher
Zeta Co    0.333
Omega      0.333
 Acme      0.167
acme       0.167
Name: proportion, dtype: float64


▸ 5 · Memoria por columna (bytes)


Index        132
Name          88
Publisher     91
Year         282
Genres       132
Sales         56
dtype: int64


  Dos detalles que se pasan por alto:

  · value_counts() DESCARTA los nulos por defecto. Si quieres saber cuántos
    registros no tienen productor, necesitas dropna=False. Es una fuente
    clásica de "me faltan filas y no sé dónde".
  · memory_usage necesita deep=True para medir el texto de verdad; sin él solo
    cuenta los punteros y te subestima el tamaño real.


In [38]:
def perfilar(df: pd.DataFrame, nombre: str = "dataset") -> pd.DataFrame:
    """Perfil de CUALQUIER DataFrame en una sola tabla.

    Esta es la función que te pedí escribir para 01_perfilar.py. La diferencia
    con imprimir cosas sueltas: devuelve una TABLA, así que se puede ordenar,
    filtrar y comparar entre las tres fuentes.
    """
    resumen = pd.DataFrame({
        "dtype":   df.dtypes.astype(str),                   # tipo, como texto legible
        "nulos":   df.isna().sum(),                         # cuántos faltan
        "nulos_%": (df.isna().mean() * 100).round(1),       # y qué porcentaje es
        "únicos":  df.nunique(dropna=True),                 # cardinalidad (sin contar NaN)
    })
    # "cardinalidad relativa": únicos / filas. Cerca de 1 = identificador;
    # cerca de 0 = categoría con pocos valores. Distingue un ID de un género.
    resumen["únicos_/_filas"] = (resumen["únicos"] / len(df)).round(3)
    # Un ejemplo real de cada columna ayuda a entender el formato de un vistazo.
    resumen["ejemplo"] = [df[c].dropna().iloc[0] if df[c].notna().any() else None
                          for c in df.columns]
    print(f"=== {nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas | "
          f"{df.memory_usage(deep=True).sum() / 1e6:.2f} MB | "
          f"{df.duplicated().sum()} duplicadas ===")
    return resumen

display(perfilar(toy, "toy"))
print("""
🎯 Para tu proyecto
   Corre perfilar() sobre las tres fuentes y compara. Lo que estás buscando:
   · columna candidata a PRODUCTOR -> cardinalidad media (cientos), pocos nulos
   · columna candidata a GÉNERO    -> cardinalidad baja (decenas)
   · si 'únicos_/_filas' ≈ 1.0, esa columna es un ID, no una categoría

   Y la prueba de dataset sintético: mira los 15 productores más frecuentes. Si
   todos tienen conteos parecidos, los datos son inventados. Los datos reales
   siguen una ley de potencias: unos pocos productores concentran casi todo.""")

=== toy: 7 filas × 5 columnas | 0.00 MB | 1 duplicadas ===


,dtype,nulos,nulos_%,únicos,únicos_/_filas,ejemplo
Name,str,1,14.3,5,0.714,Alpha
Publisher,str,1,14.3,4,0.571,Acme
Year,object,1,14.3,5,0.714,2001
Genres,str,1,14.3,5,0.714,"Action, Fantasy"
Sales,float64,0,0.0,6,0.857,1.5



🎯 Para tu proyecto
   Corre perfilar() sobre las tres fuentes y compara. Lo que estás buscando:
   · columna candidata a PRODUCTOR -> cardinalidad media (cientos), pocos nulos
   · columna candidata a GÉNERO    -> cardinalidad baja (decenas)
   · si 'únicos_/_filas' ≈ 1.0, esa columna es un ID, no una categoría

   Y la prueba de dataset sintético: mira los 15 productores más frecuentes. Si
   todos tienen conteos parecidos, los datos son inventados. Los datos reales
   siguen una ley de potencias: unos pocos productores concentran casi todo.


---

<a id="sec-3"></a>
## 3. Seleccionar y filtrar
<sub>[↑ índice](#indice)</sub>

Dos cosas que se confunden siempre y conviene fijar de entrada:

| | `.loc` | `.iloc` |
|---|---|---|
| Se indexa por | **etiqueta** (el valor del índice) | **posición** (0, 1, 2…) |
| El final del rango | **se incluye** | **no se incluye** |

`toy.loc[0:2]` devuelve **3** filas; `toy.iloc[0:2]` devuelve **2**. No es un capricho:
`.loc` usa etiquetas, y en un rango de etiquetas el final es parte del rango, como en
"de la página 10 a la 12".

In [39]:
ver("Una columna -> Series (1 dimensión)", toy["Name"])
ver("Varias columnas -> DataFrame (2 dimensiones)", toy[["Name", "Publisher"]])

ver(".loc[0:2] por ETIQUETA -> incluye la fila 2 (3 filas)", toy.loc[0:2, ["Name", "Year"]])
ver(".iloc[0:2] por POSICIÓN -> excluye la 2 (2 filas)", toy.iloc[0:2, 0:2])

# Filtro booleano: la condición produce una Series de True/False que indexa el DataFrame
mascara = toy["Sales"] > 1                       # Series de booleanos, una por fila
ver("La máscara en sí (True/False por fila)", mascara)
ver("El DataFrame filtrado con esa máscara", toy.loc[mascara])

# Combinar condiciones: & (y), | (o), ~ (no). LOS PARÉNTESIS SON OBLIGATORIOS
# porque & tiene más precedencia que > en Python.
ver("Dos condiciones con &", toy[(toy["Sales"] > 1) & (toy["Year"].notna())])
ver("Pertenencia a una lista con isin", toy[toy["Publisher"].isin(["Omega", "Zeta Co"])])
ver("Rango con between (ambos extremos incluidos)", toy[toy["Sales"].between(0.5, 3)])
ver("El mismo filtro escrito como texto, con query", toy.query("Sales > 1 and Publisher == 'Omega'"))

ver("Quitar columnas", toy.drop(columns=["Sales"]).head(3))
ver("Columnas cuyo nombre contiene 'Gen'", toy.filter(like="Gen").head(3))
print("""
  ⚠️ El error de precedencia: toy[toy["Sales"] > 1 & toy["Year"].notna()] falla
  con un ValueError críptico, porque Python evalúa 1 & toy[...] antes que >.
  Regla: cada condición entre paréntesis, siempre.""")


▸ Una columna -> Series (1 dimensión)


0      Alpha
1       Beta
2      Gamma
3      Gamma
4      Delta
5    Epsilon
6        NaN
Name: Name, dtype: str


▸ Varias columnas -> DataFrame (2 dimensiones)


,Name,Publisher
0,Alpha,Acme
1,Beta,acme
2,Gamma,Zeta Co
3,Gamma,Zeta Co
4,Delta,Omega
5,Epsilon,NaN
6,NaN,Omega



▸ .loc[0:2] por ETIQUETA -> incluye la fila 2 (3 filas)


...


▸ Quitar columnas


,Name,Publisher,Year,Genres
0,Alpha,Acme,2001,"Action, Fantasy"
1,Beta,acme,2005,Action
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action"



▸ Columnas cuyo nombre contiene 'Gen'


,Genres
0,"Action, Fantasy"
1,Action
2,"Sci-Fi, Drama, Action"



  ⚠️ El error de precedencia: toy[toy["Sales"] > 1 & toy["Year"].notna()] falla
  con un ValueError críptico, porque Python evalúa 1 & toy[...] antes que >.
  Regla: cada condición entre paréntesis, siempre.


---

<a id="sec-4"></a>
## 4. Limpieza y tipos
<sub>[↑ índice](#indice)</sub>

> ⚠️ **Antes de limpiar, decide qué significa "limpio".** Las reglas de limpieza salen
> del **modelo destino**, no al revés. Si todavía no sabes si un stand es la editorial
> o la revista, no sabes qué columna es tu `stand_id`, y por lo tanto no sabes si una
> fila sin editorial se descarta o se asigna a "Independiente".
>
> Esa es una decisión del grupo, y hay que tomarla **antes** de escribir el script.

In [40]:
df = toy.copy()                 # .copy() SIEMPRE antes de modificar: evita SettingWithCopyWarning
print(f"inicio: {len(df)} filas")

df = df.dropna(subset=["Publisher"])      # fuera las filas sin productor (no sirven como stand)
print(f"tras dropna(Publisher): {len(df)} filas")

df = df.drop_duplicates()                 # fuera duplicados EXACTOS (todas las columnas iguales)
print(f"tras drop_duplicates(): {len(df)} filas")

df = df.drop_duplicates(subset=["Name"], keep="first")   # un título una vez; nos quedamos con el 1º
print(f"tras drop_duplicates(subset=['Name']): {len(df)} filas")

df["Genres"] = df["Genres"].fillna("")    # nulo -> cadena vacía (para poder usar .str después)

# to_numeric con errors="coerce": lo que no se puede convertir pasa a NaN en vez de reventar.
# Es la diferencia entre "el script falla" y "el script te deja un NaN que puedes contar".
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Year"] = df["Year"].astype("Int64")   # Int64 con I MAYÚSCULA: entero que ADMITE nulos

df = df.rename(columns={"Name": "nombre", "Publisher": "productor"})   # nombres propios del proyecto
df = df.reset_index(drop=True)            # reindexar 0..n-1; drop=True descarta el índice viejo

ver("Resultado limpio", df)
ver("Tipos finales", df.dtypes)
print("""
  Fíjate en Int64 vs int64: el int64 de NumPy NO admite nulos, así que si hay un
  NaN la columna se queda en float y los años se imprimen como 2001.0. El Int64
  de pandas (con mayúscula) sí admite <NA> y mantiene el año como entero.""")

inicio: 7 filas
tras dropna(Publisher): 6 filas
tras drop_duplicates(): 5 filas
tras drop_duplicates(subset=['Name']): 5 filas

▸ Resultado limpio


,nombre,productor,Year,Genres,Sales
0,Alpha,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Delta,Omega,<NA>,,0.1
4,NaN,Omega,2020,Drama,0.7



▸ Tipos finales


nombre           str
productor        str
Year           Int64
Genres           str
Sales        float64
dtype: object


  Fíjate en Int64 vs int64: el int64 de NumPy NO admite nulos, así que si hay un
  NaN la columna se queda en float y los años se imprimen como 2001.0. El Int64
  de pandas (con mayúscula) sí admite <NA> y mantiene el año como entero.


In [41]:
s = pd.Series(["a", "b", "c", "a"])          # serie de ejemplo para las transformaciones

ver("replace: cambia valores concretos", s.replace({"a": "A"}))
ver("map con dict: lo NO encontrado -> NaN", s.map({"a": 1, "b": 2}))
ver("apply: una función por elemento", s.apply(lambda x: x.upper()))
ver("astype('category'): ahorra memoria con pocos valores distintos", s.astype("category"))
ver("memoria como texto vs como categoría",
    {"texto": int(s.memory_usage(deep=True)),
     "categoría": int(s.astype("category").memory_usage(deep=True))})

ver("to_datetime: texto -> fecha", pd.to_datetime(["2026-05-01 10:00", "2026-05-01 12:30"]))
ver("aritmética de fechas con Timedelta", pd.to_datetime("2026-05-01") + pd.Timedelta(hours=2))
ver("cut: convertir números en categorías ordenadas",
    pd.cut(pd.Series([1, 5, 9]), bins=[0, 3, 6, 10], labels=["bajo", "medio", "alto"]))
ver("clip: recortar valores fuera de rango", pd.Series([3, 1, 2]).clip(lower=2))
print("""
  ⚠️ map vs replace, que no son lo mismo:
     map    -> lo que NO está en el diccionario se convierte en NaN
     replace-> lo que NO está se queda igual
  Para una tabla de mapeo de etiquetas (§7) normalmente quieres map, porque ese
  NaN es justamente la señal de "esta etiqueta no la tengo mapeada todavía".""")


▸ replace: cambia valores concretos


0    A
1    b
2    c
3    A
dtype: str


▸ map con dict: lo NO encontrado -> NaN


0    1.0
1    2.0
2    NaN
3    1.0
dtype: float64


▸ apply: una función por elemento


...


▸ memoria como texto vs como categoría
{'texto': 168, 'categoría': 164}

▸ to_datetime: texto -> fecha
DatetimeIndex(['2026-05-01 10:00:00', '2026-05-01 12:30:00'], dtype='datetime64[us]', freq=None)

▸ aritmética de fechas con Timedelta
2026-05-01 02:00:00

▸ cut: convertir números en categorías ordenadas


0     bajo
1    medio
2     alto
dtype: category
Categories (3, str): ['bajo' < 'medio' < 'alto']


▸ clip: recortar valores fuera de rango


0    3
1    2
2    2
dtype: int64


  ⚠️ map vs replace, que no son lo mismo:
     map    -> lo que NO está en el diccionario se convierte en NaN
     replace-> lo que NO está se queda igual
  Para una tabla de mapeo de etiquetas (§7) normalmente quieres map, porque ese
  NaN es justamente la señal de "esta etiqueta no la tengo mapeada todavía".


---

<a id="sec-5"></a>
## 5. Texto con `.str`
<sub>[↑ índice](#indice)</sub>

### Por qué esta sección decide si tu recomendador sirve

`"Acme"`, `" Acme "` y `"acme"` son **tres productores distintos** para pandas. Si no
los unificas antes de agrupar, tu feria tendrá tres stands donde debería haber uno, y
las etiquetas de ese productor quedarán repartidas entre los tres.

El accesor `.str` aplica operaciones de texto a toda una columna de una vez, y maneja
los nulos sin reventar (los propaga como `NaN`).

In [42]:
t = pd.Series([" Role-Playing ", "SCI-FI", "Slice of Life", None, "Sports/Racing"])
ver("Serie original (fíjate en el None)", t)

ver("strip: quita espacios de los extremos", t.str.strip())
ver("lower: todo a minúsculas", t.str.lower())
ver("replace literal (regex=False)", t.str.replace("-", " ", regex=False))
ver("replace con regex: colapsar espacios múltiples", t.str.replace(r"\s+", " ", regex=True))
ver("contains: ¿contiene 'sci'? (na=False evita el error en filtros)",
    t.str.contains("sci", case=False, na=False))
ver("startswith", t.str.startswith("Sl", na=False))
ver("len: largo de cada texto (NaN se propaga)", t.str.len())
ver("split: texto -> lista", t.str.split("/"))
ver("extract: capturar con regex (devuelve DataFrame)", t.str.extract(r"(\w+)"))
ver("cat: unir toda la columna en un solo texto", t.str.cat(sep=" | ", na_rep="∅"))
print("""
  ⚠️ na=False en contains/startswith no es opcional cuando filtras. Sin él, los
  nulos producen NaN en vez de False, y pandas no sabe indexar con NaN: el
  filtro lanza "Cannot mask with non-boolean array containing NA".""")


▸ Serie original (fíjate en el None)


0     Role-Playing 
1            SCI-FI
2     Slice of Life
3               NaN
4     Sports/Racing
dtype: str


▸ strip: quita espacios de los extremos


0     Role-Playing
1           SCI-FI
2    Slice of Life
3              NaN
4    Sports/Racing
dtype: str


▸ lower: todo a minúsculas


...


▸ split: texto -> lista


0    [ Role-Playing ]
1            [SCI-FI]
2     [Slice of Life]
3                 NaN
4    [Sports, Racing]
dtype: object


▸ extract: capturar con regex (devuelve DataFrame)


,0
0,Role
1,SCI
2,Slice
3,NaN
4,Sports



▸ cat: unir toda la columna en un solo texto
 Role-Playing  | SCI-FI | Slice of Life | ∅ | Sports/Racing

  ⚠️ na=False en contains/startswith no es opcional cuando filtras. Sin él, los
  nulos producen NaN en vez de False, y pandas no sabe indexar con NaN: el
  filtro lanza "Cannot mask with non-boolean array containing NA".


In [43]:
def quitar_tildes(x):
    """'Acción' -> 'Accion', sin tabla de reemplazos a mano.

    NFKD descompone cada letra acentuada en (letra base + marca de acento);
    unicodedata.combining() identifica esas marcas, y las descartamos. Hacerlo
    con .replace("á","a") obliga a enumerar todos los casos y se olvida alguno.
    """
    if not isinstance(x, str):      # los nulos y números pasan de largo sin romper
        return x
    descompuesto = unicodedata.normalize("NFKD", x)              # 'ó' -> 'o' + '´'
    return "".join(c for c in descompuesto if not unicodedata.combining(c))   # bota los '´'


def normalizar(serie: pd.Series) -> pd.Series:
    """La función que debe aplicarse a TODA etiqueta antes de compararla.

    El orden importa: tildes -> minúsculas -> separadores. Si pusieras el
    reemplazo de separadores antes de quitar tildes, 'Ciencia-Ficción' y
    'ciencia ficcion' no colapsarían en lo mismo.
    """
    return (serie.map(quitar_tildes)                    # 1. fuera las tildes
                 .str.strip()                           # 2. fuera espacios de los bordes
                 .str.lower()                           # 3. todo a minúsculas
                 .str.replace(r"[\s_/]+", "-", regex=True)   # 4. separadores -> guion único
                 .str.strip("-"))                       # 5. sin guiones sobrantes en los bordes


ver("quitar_tildes sobre una columna", pd.Series(["Acción", "Ciencia ficción"]).map(quitar_tildes))
ver("normalizar() completo", normalizar(t))

# La prueba que importa: ¿cuántas variantes colapsan en una sola forma?
variantes = pd.Series(["Ciencia-Ficción", "ciencia ficcion", "  CIENCIA   FICCION  ",
                       "Ciencia_Ficción", "ciencia.ficción"])
normalizadas = normalizar(variantes)
ver("5 variantes de lo mismo", normalizadas.tolist())
ver("¿colapsan en UNA sola?", normalizadas.nunique() == 1)
print("""
  ⚠️ El punto de 'ciencia.ficción' NO lo cubre el regex [\\s_/]+. Por eso el
  resultado tiene 2 formas distintas y no 1. Es un ejemplo deliberado: la
  normalización nunca está "terminada", se completa cuando ves los datos reales.

🎯 Para tu proyecto
   Escribe TU función normalizar() y pásala por todas las etiquetas de las tres
   fuentes. Después cuenta: ¿cuántas etiquetas distintas había antes y cuántas
   quedan? Si el número no baja, la normalización no está haciendo nada.""")


▸ quitar_tildes sobre una columna


0             Accion
1    Ciencia ficcion
dtype: str


▸ normalizar() completo


0     role-playing
1           sci-fi
2    slice-of-life
3              NaN
4    sports-racing
dtype: str


▸ 5 variantes de lo mismo
['ciencia-ficcion', 'ciencia-ficcion', 'ciencia-ficcion', 'ciencia-ficcion', 'ciencia.ficcion']

▸ ¿colapsan en UNA sola?
False

  ⚠️ El punto de 'ciencia.ficción' NO lo cubre el regex [\s_/]+. Por eso el
  resultado tiene 2 formas distintas y no 1. Es un ejemplo deliberado: la
  normalización nunca está "terminada", se completa cuando ves los datos reales.

🎯 Para tu proyecto
   Escribe TU función normalizar() y pásala por todas las etiquetas de las tres
   fuentes. Después cuenta: ¿cuántas etiquetas distintas había antes y cuántas
   quedan? Si el número no baja, la normalización no está haciendo nada.


---

<a id="sec-6"></a>
## 6. Columnas con varios valores
<sub>[↑ índice](#indice)</sub>

Aquí está la medición que te mencioné en §2: **cuántas etiquetas trae cada ítem**.

```
FORMA ANCHA (como viene)              FORMA LARGA (para contar y cruzar)
┌───────┬────────────────────┐        ┌───────┬──────────┐
│ Gamma │ Sci-Fi, Drama,     │        │ Gamma │ sci-fi   │
│       │ Action             │   →    │ Gamma │ drama    │
└───────┴────────────────────┘        │ Gamma │ action   │
  1 fila, 3 valores en un string      └───────┴──────────┘
                                        3 filas, 1 valor cada una
```

`explode` es el puente: convierte cada elemento de una lista en su propia fila. Y
`groupby(...).agg(list)` es el puente de vuelta.

In [13]:
g = toy[["Name", "Genres"]].dropna().drop_duplicates().copy()   # base limpia para el ejemplo
ver("Punto de partida", g)

# Paso 1: el string se parte en una lista de Python
g["lista"] = g["Genres"].str.split(",")
ver("1 · split -> columna de LISTAS", g[["Name", "lista"]])

# Paso 2: limpiar CADA elemento de cada lista.
# map recorre filas; la list comprehension recorre los elementos dentro de la fila.
g["lista"] = g["lista"].map(lambda xs: [x.strip().lower() for x in xs])
ver("2 · cada elemento limpio", g[["Name", "lista"]])

# LA MEDICIÓN QUE DECIDE SI JACCARD SIRVE
n_tags = g["lista"].map(len)
ver("3 · etiquetas por ítem", {"mediana": float(n_tags.median()), "máximo": int(n_tags.max()),
                               "mínimo": int(n_tags.min())})

# Paso 4: explode -> una fila por etiqueta
largo = g.explode("lista").rename(columns={"lista": "genero"})
ver("4 · forma larga (fíjate en el índice REPETIDO)", largo[["Name", "genero"]])
print("""
  ⚠️ explode conserva el índice original, así que quedan índices repetidos
  (0, 0, 1, 2, 2, 2...). Si eso te molesta para un merge posterior, agrega
  .reset_index(drop=True).

  Y la medición del paso 3 es la que te dije que decide el proyecto: si la
  MEDIANA de etiquetas por ítem es 1, Jaccard entre dos ítems solo puede valer
  0 o 1, y deja de ser una medida de similitud para volverse una igualdad.""")


▸ Punto de partida


,Name,Genres
0,Alpha,"Action, Fantasy"
1,Beta,Action
2,Gamma,"Sci-Fi, Drama, Action"
5,Epsilon,Fantasy



▸ 1 · split -> columna de LISTAS


,Name,lista
0,Alpha,"[Action, Fantasy]"
1,Beta,[Action]
2,Gamma,"[Sci-Fi, Drama, Action]"
5,Epsilon,[Fantasy]



▸ 2 · cada elemento limpio


,Name,lista
0,Alpha,"[action, fantasy]"
1,Beta,[action]
2,Gamma,"[sci-fi, drama, action]"
5,Epsilon,[fantasy]



▸ 3 · etiquetas por ítem
{'mediana': 1.5, 'máximo': 3, 'mínimo': 1}

▸ 4 · forma larga (fíjate en el índice REPETIDO)


,Name,genero
0,Alpha,action
0,Alpha,fantasy
1,Beta,action
2,Gamma,sci-fi
2,Gamma,drama
2,Gamma,action
5,Epsilon,fantasy



  ⚠️ explode conserva el índice original, así que quedan índices repetidos
  (0, 0, 1, 2, 2, 2...). Si eso te molesta para un merge posterior, agrega
  .reset_index(drop=True).

  Y la medición del paso 3 es la que te dije que decide el proyecto: si la
  MEDIANA de etiquetas por ítem es 1, Jaccard entre dos ítems solo puede valer
  0 o 1, y deja de ser una medida de similitud para volverse una igualdad.


In [14]:
ver("get_dummies sobre el string: matriz 0/1 por género",
    g["Genres"].str.get_dummies(sep=", "))
ver("get_dummies sobre la forma larga", pd.get_dummies(largo["genero"]).head())
ver("Frecuencia de cada género", largo["genero"].value_counts())

# Caso frecuente en JSON: la columna es TEXTO que PARECE una lista
como_texto = pd.Series(["['a', 'b']", "['c']"])
ver("Texto que parece lista (type)", como_texto.map(type).tolist())
ver("ast.literal_eval lo convierte en lista real", como_texto.map(ast.literal_eval))
ver("...y ahora sí tiene tipo list", como_texto.map(ast.literal_eval).map(type).tolist())

# Diccionarios anidados (lo típico de un JSON): una columna por clave
anidado = pd.DataFrame({"id": [1, 2], "info": [{"x": 1, "y": 2}, {"x": 3, "y": 4}]})
ver("Columna con dicts dentro", anidado)
ver("json_normalize: dict -> columnas", pd.json_normalize(anidado["info"]))
print("""
  ⚠️ NUNCA uses eval() para esto. ast.literal_eval solo acepta literales de
  Python (listas, dicts, números, strings) y por eso es seguro; eval ejecuta
  cualquier código que venga en el archivo.

🎯 Para tu proyecto
   La columna popular_shelves de Goodreads es una lista de diccionarios que
   MEZCLA géneros con ruido de usuarios ('to-read', 'currently-reading',
   'favorites'). Antes de explotarla tienes que decidir el criterio de filtrado:
   ¿por conteo mínimo?, ¿por lista blanca de géneros? Esa decisión es tuya y hay
   que poder justificarla.""")


▸ get_dummies sobre el string: matriz 0/1 por género


,Action,Drama,Fantasy,Sci-Fi
0,1,0,1,0
1,1,0,0,0
2,1,1,0,1
5,0,0,1,0



▸ get_dummies sobre la forma larga


,action,drama,fantasy,sci-fi
0,True,False,False,False
0,False,False,True,False
1,True,False,False,False
2,False,False,False,True
2,False,True,False,False



▸ Frecuencia de cada género


...


▸ ...y ahora sí tiene tipo list
[<class 'list'>, <class 'list'>]

▸ Columna con dicts dentro


,id,info
0,1,"{'x': 1, 'y': 2}"
1,2,"{'x': 3, 'y': 4}"



▸ json_normalize: dict -> columnas


,x,y
0,1,2
1,3,4



  ⚠️ NUNCA uses eval() para esto. ast.literal_eval solo acepta literales de
  Python (listas, dicts, números, strings) y por eso es seguro; eval ejecuta
  cualquier código que venga en el archivo.

🎯 Para tu proyecto
   La columna popular_shelves de Goodreads es una lista de diccionarios que
   MEZCLA géneros con ruido de usuarios ('to-read', 'currently-reading',
   'favorites'). Antes de explotarla tienes que decidir el criterio de filtrado:
   ¿por conteo mínimo?, ¿por lista blanca de géneros? Esa decisión es tuya y hay
   que poder justificarla.


---

<a id="sec-7"></a>
## 7. Vocabulario común y tabla de mapeo
<sub>[↑ índice](#indice)</sub>

Normalizar (§5) resuelve la **ortografía**; no resuelve el **vocabulario**. `"sci-fi"`,
`"science fiction"` y `"cf"` pueden estar los tres perfectamente normalizados y seguir
siendo tres etiquetas distintas.

> **Idea clave, y es de diseño, no de código.** La tabla de equivalencias es un
> **artefacto humano**: un CSV que una persona del grupo edita y que el código
> solo lee y aplica. Dos razones concretas: Cris y Sofi pueden discutirla sin tocar
> Python, y en la defensa puedes **mostrarla** como decisión de diseño. Un
> `if genero == "isekai": return "fantasia"` enterrado en una función no se le puede
> mostrar a nadie.

In [15]:
# La tabla de mapeo. En el proyecto real esto es un CSV que se lee con read_csv.
mapeo = pd.DataFrame({
    "origen": ["sci-fi", "science fiction", "role-playing", "fantasy", "action"],
    "tag":    ["ciencia-ficcion", "ciencia-ficcion", "rol-fantasia", "rol-fantasia", "accion"],
    "nivel":  ["general", "general", "general", "especifico", "general"],
})
ver("La tabla de equivalencias", mapeo)

# --- Opción A: dict + map. Rápido, y NO puede duplicar filas. ---
dic = dict(zip(mapeo["origen"], mapeo["tag"]))     # dos columnas -> diccionario
ver("El diccionario resultante", dic)
largo["tag"] = largo["genero"].map(dic)            # lo no encontrado queda NaN
ver("Resultado con map", largo[["Name", "genero", "tag"]])

# --- Opción B: merge. Permite traer VARIAS columnas de la tabla de mapeo. ---
con_tag = (largo.drop(columns="tag")                                  # quitamos la de la opción A
                .merge(mapeo, left_on="genero", right_on="origen", how="left"))
ver("Resultado con merge (trae también 'nivel')", con_tag[["Name", "genero", "tag", "nivel"]])

# --- EL CONTROL DE CALIDAD: qué quedó sin mapear ---
sin_mapear = largo.loc[largo["tag"].isna(), "genero"].value_counts()
ver("Géneros SIN equivalencia (hay que decidir qué hacer)", sin_mapear)
ver("Cobertura de la tabla de mapeo", f"{largo['tag'].notna().mean():.0%}")
print("""
  Esa última tabla es lo más valioso de la sección. Un ETL que mapea en silencio
  y descarta lo que no reconoce acumula basura durante semanas. Uno que REPORTA
  lo no mapeado te obliga a decidir cada caso: ¿sinónimo que falta?, ¿etiqueta
  nueva?, ¿error de tipeo que hay que rechazar?

  A vs B, para que elijas con criterio:
    map   -> una columna, no puede duplicar filas. Úsalo por defecto.
    merge -> varias columnas, pero si la tabla derecha tiene claves REPETIDAS
             multiplica filas sin avisar. Verifica len() antes y después.""")


▸ La tabla de equivalencias


,origen,tag,nivel
0,sci-fi,ciencia-ficcion,general
1,science fiction,ciencia-ficcion,general
2,role-playing,rol-fantasia,general
3,fantasy,rol-fantasia,especifico
4,action,accion,general



▸ El diccionario resultante
{'sci-fi': 'ciencia-ficcion', 'science fiction': 'ciencia-ficcion', 'role-playing': 'rol-fantasia', 'fantasy': 'rol-fantasia', 'action': 'accion'}

▸ Resultado con map


,Name,genero,tag
0,Alpha,action,accion
0,Alpha,fantasy,rol-fantasia
1,Beta,action,accion
2,Gamma,sci-fi,ciencia-ficcion
2,Gamma,drama,NaN
2,Gamma,action,accion
5,Epsilon,fantasy,rol-fantasia



▸ Resultado con merge (trae también 'nivel')


,Name,genero,tag,nivel
0,Alpha,action,accion,general
1,Alpha,fantasy,rol-fantasia,especifico
2,Beta,action,accion,general
3,Gamma,sci-fi,ciencia-ficcion,general
4,Gamma,drama,NaN,NaN
5,Gamma,action,accion,general
6,Epsilon,fantasy,rol-fantasia,especifico



▸ Géneros SIN equivalencia (hay que decidir qué hacer)


genero
drama    1
Name: count, dtype: int64


▸ Cobertura de la tabla de mapeo
86%

  Esa última tabla es lo más valioso de la sección. Un ETL que mapea en silencio
  y descarta lo que no reconoce acumula basura durante semanas. Uno que REPORTA
  lo no mapeado te obliga a decidir cada caso: ¿sinónimo que falta?, ¿etiqueta
  nueva?, ¿error de tipeo que hay que rechazar?

  A vs B, para que elijas con criterio:
    map   -> una columna, no puede duplicar filas. Úsalo por defecto.
    merge -> varias columnas, pero si la tabla derecha tiene claves REPETIDAS
             multiplica filas sin avisar. Verifica len() antes y después.


In [16]:
# Agrupar orígenes por tag: así se construye el campo "orígenes" de la colección tags
ver("tag -> lista de sus orígenes", mapeo.groupby("tag")["origen"].agg(list))
ver("tag -> cuántos orígenes tiene", mapeo.groupby("tag")["origen"].nunique())

# La tabla se guarda y se lee como CSV, que es lo que la hace editable por el grupo
buffer = StringIO()                                  # simulamos el archivo
mapeo.to_csv(buffer, index=False)                    # index=False: no guardar 0,1,2...
buffer.seek(0)                                       # rebobinar para poder leer
ver("La misma tabla tras ir y volver de CSV", pd.read_csv(buffer))
print("""
🎯 Para tu proyecto
   Tu tabla de mapeo es la pieza que permite el CRUCE ENTRE DOMINIOS: si un juego
   de 'fantasy' y un manga de 'isekai' no caen bajo el mismo tag general, no hay
   recomendación cruzada y el proyecto pierde su justificación.

   Decisión pendiente del grupo: ¿cuántos niveles generales? Con menos de 3 todo
   cae junto y el cruce no discrimina; con más de ~8 ningún par comparte general
   y el cruce desaparece.""")


▸ tag -> lista de sus orígenes


tag
accion                              [action]
ciencia-ficcion    [sci-fi, science fiction]
rol-fantasia         [role-playing, fantasy]
Name: origen, dtype: object


▸ tag -> cuántos orígenes tiene


tag
accion             1
ciencia-ficcion    2
rol-fantasia       2
Name: origen, dtype: int64


▸ La misma tabla tras ir y volver de CSV


,origen,tag,nivel
0,sci-fi,ciencia-ficcion,general
1,science fiction,ciencia-ficcion,general
2,role-playing,rol-fantasia,general
3,fantasy,rol-fantasia,especifico
4,action,accion,general



🎯 Para tu proyecto
   Tu tabla de mapeo es la pieza que permite el CRUCE ENTRE DOMINIOS: si un juego
   de 'fantasy' y un manga de 'isekai' no caen bajo el mismo tag general, no hay
   recomendación cruzada y el proyecto pierde su justificación.

   Decisión pendiente del grupo: ¿cuántos niveles generales? Con menos de 3 todo
   cae junto y el cruce no discrimina; con más de ~8 ningún par comparte general
   y el cruce desaparece.


---

<a id="sec-8"></a>
## 8. Agrupar y agregar: construir los stands
<sub>[↑ índice](#indice)</sub>

El modelo del proyecto dice que **un stand es un productor**. Eso significa pasar de
una tabla de títulos a una tabla de productores, **juntando las etiquetas de todo su
catálogo** sin repetirlas.

`groupby` sigue el patrón *split–apply–combine*: parte por clave, aplica una función a
cada grupo, y recombina.

In [17]:
base = con_tag.copy()
# Asignamos un productor a cada título (en el proyecto real viene en los datos)
base["Publisher"] = base["Name"].map({"Alpha": "acme", "Beta": "acme",
                                      "Gamma": "zeta co", "Epsilon": "omega"})
base = base.dropna(subset=["Publisher"])        # sin productor no hay stand
ver("Base: una fila por (título, etiqueta)", base[["Name", "Publisher", "tag"]])

# AGREGACIÓN NOMBRADA: nombre_salida=(columna_origen, función)
stands = (base.groupby("Publisher")                     # 1. partir por productor
              .agg(n_titulos=("Name", "nunique"),       # 2a. títulos distintos del stand
                   # 2b. todas sus etiquetas, sin repetir y ordenadas.
                   #     set() quita duplicados; sorted() hace el resultado determinista
                   #     (sin él, el orden del set cambia entre ejecuciones)
                   tags=("tag", lambda s: sorted(set(s.dropna()))))
              .reset_index())                           # 3. el grupo vuelve a ser columna
ver("Los stands construidos", stands)
ver("Comprobación: ¿alguna etiqueta repetida dentro de un stand?",
    bool(stands["tags"].map(lambda t: len(t) == len(set(t))).all()))
print("""
  Dos detalles que valen una pregunta de defensa:

  · sorted(set(...)) no es decoración. set() garantiza que un stand no tenga la
    misma etiqueta dos veces (lo que inflaría artificialmente su similitud);
    sorted() garantiza que dos corridas den EXACTAMENTE el mismo resultado.
  · reset_index() es necesario porque groupby deja la clave como ÍNDICE. Si lo
    olvidas, 'Publisher' no es una columna y el merge posterior falla con un
    KeyError que confunde.""")


▸ Base: una fila por (título, etiqueta)


,Name,Publisher,tag
0,Alpha,acme,accion
1,Alpha,acme,rol-fantasia
2,Beta,acme,accion
3,Gamma,zeta co,ciencia-ficcion
4,Gamma,zeta co,NaN
5,Gamma,zeta co,accion
6,Epsilon,omega,rol-fantasia



▸ Los stands construidos


,Publisher,n_titulos,tags
0,acme,2,"[accion, rol-fantasia]"
1,omega,1,[rol-fantasia]
2,zeta co,1,"[accion, ciencia-ficcion]"



▸ Comprobación: ¿alguna etiqueta repetida dentro de un stand?
True

  Dos detalles que valen una pregunta de defensa:

  · sorted(set(...)) no es decoración. set() garantiza que un stand no tenga la
    misma etiqueta dos veces (lo que inflaría artificialmente su similitud);
    sorted() garantiza que dos corridas den EXACTAMENTE el mismo resultado.
  · reset_index() es necesario porque groupby deja la clave como ÍNDICE. Si lo
    olvidas, 'Publisher' no es una columna y el merge posterior falla con un
    KeyError que confunde.


In [18]:
ver("size: filas por grupo", base.groupby("Publisher").size())
ver("nunique: etiquetas distintas por grupo", base.groupby("Publisher")["tag"].nunique())
ver("Tabla cruzada con groupby + unstack",
    base.groupby(["Publisher", "tag"]).size().unstack(fill_value=0))
ver("Lo mismo, más directo, con crosstab", pd.crosstab(base["Publisher"], base["tag"]))
ver("Varias agregaciones a la vez", base.groupby("Publisher")["tag"].agg(["count", "nunique"]))

# transform: devuelve un valor POR FILA, no por grupo. Útil para añadir columnas.
base["tags_del_stand"] = base.groupby("Publisher")["tag"].transform("count")
ver("transform: el dato del grupo pegado a cada fila", base[["Name", "Publisher", "tags_del_stand"]])
ver("pivot_table, el equivalente de la tabla dinámica de Excel",
    base.pivot_table(index="Publisher", columns="tag", values="Name",
                     aggfunc="count", fill_value=0))
print("""
  ⚠️ agg vs transform, que es la confusión clásica:
     agg       -> una fila POR GRUPO   (3 productores -> 3 filas)
     transform -> una fila POR FILA    (7 filas -> 7 filas, con el valor del grupo)
  Si una agregación te devuelve menos filas de las que esperabas, querías transform.""")


▸ size: filas por grupo


Publisher
acme       3
omega      1
zeta co    3
dtype: int64


▸ nunique: etiquetas distintas por grupo


Publisher
acme       2
omega      1
zeta co    2
Name: tag, dtype: int64


▸ Tabla cruzada con groupby + unstack


...


▸ transform: el dato del grupo pegado a cada fila


,Name,Publisher,tags_del_stand
0,Alpha,acme,3
1,Alpha,acme,3
2,Beta,acme,3
3,Gamma,zeta co,2
4,Gamma,zeta co,2
5,Gamma,zeta co,2
6,Epsilon,omega,1



▸ pivot_table, el equivalente de la tabla dinámica de Excel


tag,accion,ciencia-ficcion,rol-fantasia
Publisher,,,
acme,2,0,1
omega,0,0,1
zeta co,1,1,0



  ⚠️ agg vs transform, que es la confusión clásica:
     agg       -> una fila POR GRUPO   (3 productores -> 3 filas)
     transform -> una fila POR FILA    (7 filas -> 7 filas, con el valor del grupo)
  Si una agregación te devuelve menos filas de las que esperabas, querías transform.


---

<a id="sec-9"></a>
## 9. Unir tablas
<sub>[↑ índice](#indice)</sub>

| Operación | Qué hace | Cuándo |
|---|---|---|
| `merge` | une **por clave**, como un JOIN de SQL | añadir columnas de otra tabla |
| `concat` | **apila** tablas | juntar los stands de los 3 dominios |
| `join` | merge por índice | cuando la clave ya es el índice |

> ⚠️ **El error que más cuesta encontrar:** un `merge` que **multiplica filas**. Si la
> tabla derecha tiene la clave repetida, cada fila de la izquierda se duplica por cada
> coincidencia. Verifica `len()` antes y después, siempre.

In [19]:
a = pd.DataFrame({"k": [1, 2, 3], "x": ["a", "b", "c"]})
b = pd.DataFrame({"k": [2, 3, 4], "y": ["B", "C", "D"]})
ver("Tabla a", a); ver("Tabla b", b)

ver("inner: solo las claves que están en AMBAS", a.merge(b, on="k", how="inner"))
ver("left: todo lo de a; lo que falta de b queda NaN", a.merge(b, on="k", how="left"))
ver("outer + indicator: todo, y _merge dice de dónde vino",
    a.merge(b, on="k", how="outer", indicator=True))

# Demostración del merge que multiplica filas
b_dup = pd.DataFrame({"k": [2, 2, 3], "y": ["B1", "B2", "C"]})     # la clave 2 está DOS veces
unido = a.merge(b_dup, on="k", how="left")
ver("Tabla derecha con clave repetida", b_dup)
ver("El merge DUPLICÓ la fila k=2", unido)
ver("Control de integridad", {"filas antes": len(a), "filas después": len(unido),
                              "¿se multiplicaron?": len(unido) != len(a)})

# Apilar: los stands de los tres dominios tienen las mismas columnas
s1 = pd.DataFrame({"nombre": ["A"], "dominio": ["juego"]})
s2 = pd.DataFrame({"nombre": ["B"], "dominio": ["anime"]})
ver("concat: apilar y reindexar", pd.concat([s1, s2], ignore_index=True))
ver("concat con keys: conserva de qué tabla vino cada fila", pd.concat([s1, s2], keys=["f1", "f2"]))

# assign + pipe: encadenar sin modificar el original
ver("assign añade columnas; pipe encadena funciones propias",
    s1.assign(fuente="vgsales").pipe(lambda d: d.rename(columns={"nombre": "stand"})))
print("""
  validate= es el seguro que casi nadie usa y debería:
      a.merge(b, on="k", how="left", validate="one_to_one")
  Si la relación no es la que declaraste, lanza MergeError en vez de duplicar en
  silencio. Ponlo en tu ETL y te enteras el día uno, no en la presentación.""")


▸ Tabla a


,k,x
0,1,a
1,2,b
2,3,c



▸ Tabla b


,k,y
0,2,B
1,3,C
2,4,D



▸ inner: solo las claves que están en AMBAS


...


▸ concat con keys: conserva de qué tabla vino cada fila


,,nombre,dominio
f1,0,A,juego
f2,0,B,anime



▸ assign añade columnas; pipe encadena funciones propias


,stand,dominio,fuente
0,A,juego,vgsales



  validate= es el seguro que casi nadie usa y debería:
      a.merge(b, on="k", how="left", validate="one_to_one")
  Si la relación no es la que declaraste, lanza MergeError en vez de duplicar en
  silencio. Ponlo en tu ETL y te enteras el día uno, no en la presentación.


---

<a id="sec-10"></a>
## 10. Frecuencias, IDF y similitud
<sub>[↑ índice](#indice)</sub>

No todas las etiquetas informan lo mismo. Si `accion` está en el 80 % de los stands,
compartirla no dice casi nada; compartir `terror`, que está en el 4 %, sí.

$$\text{idf}(t)=\log\frac{N}{\text{df}(t)}\qquad\text{o, estabilizada:}\qquad
\text{idf}(t)=\log\frac{1+N}{1+\text{df}(t)}+1$$

**Por qué el logaritmo:** sin él, una etiqueta presente en 1 de 10.000 stands valdría
10.000 veces más que una presente en todos, y dominaría cualquier puntaje por sí sola.
El log comprime ese rango sin alterar el orden.

**Por qué el `+1`:** evita dividir por cero con una etiqueta que no aparece en el
corpus — el caso de una etiqueta nueva que llega de un expositor nuevo.

In [20]:
frec = pd.DataFrame({"tag": ["accion", "rol", "terror"], "frecuencia": [40, 10, 2]})
N = 50                                                  # total de stands del ejemplo

frec["idf"] = np.log(N / frec["frecuencia"])            # forma cruda
frec["idf_suave"] = np.log((1 + N) / (1 + frec["frecuencia"])) + 1   # forma estabilizada
ver("IDF: las etiquetas raras pesan más", frec.sort_values("idf", ascending=False).round(3))

ver("Ordenar por varias columnas, con direcciones distintas",
    frec.sort_values(["frecuencia", "tag"], ascending=[False, True]))
ver("nlargest: top-N sin ordenar todo (más rápido)", frec.nlargest(2, "frecuencia"))
frec["ranking"] = frec["frecuencia"].rank(ascending=False, method="dense")
ver("rank con method='dense': sin saltos tras un empate", frec.round(3))
print("""
  method= en rank() cambia qué pasa con los empates:
    'dense'   1,2,2,3   (no deja huecos)
    'min'     1,2,2,4   (deja hueco: es el ranking deportivo)
    'average' 1,2.5,2.5,4
  Elige a conciencia: el ranking de stands que muestres al asistente depende de esto.""")


▸ IDF: las etiquetas raras pesan más


,tag,frecuencia,idf,idf_suave
2,terror,2,3.219,3.833
1,rol,10,1.609,2.534
0,accion,40,0.223,1.218



▸ Ordenar por varias columnas, con direcciones distintas


,tag,frecuencia,idf,idf_suave
0,accion,40,0.223144,1.218254
1,rol,10,1.609438,2.533930
2,terror,2,3.218876,3.833213



▸ nlargest: top-N sin ordenar todo (más rápido)


,tag,frecuencia,idf,idf_suave
0,accion,40,0.223144,1.218254
1,rol,10,1.609438,2.533930



▸ rank con method='dense': sin saltos tras un empate


,tag,frecuencia,idf,idf_suave,ranking
0,accion,40,0.223,1.218,1.0
1,rol,10,1.609,2.534,2.0
2,terror,2,3.219,3.833,3.0



  method= en rank() cambia qué pasa con los empates:
    'dense'   1,2,2,3   (no deja huecos)
    'min'     1,2,2,4   (deja hueco: es el ranking deportivo)
    'average' 1,2.5,2.5,4
  Elige a conciencia: el ranking de stands que muestres al asistente depende de esto.


In [21]:
def jaccard(a, b) -> float:
    """|A ∩ B| / |A ∪ B|. Dos conjuntos idénticos dan 1; disjuntos dan 0."""
    a, b = set(a), set(b)                     # set() permite usar & (intersección) y | (unión)
    union = a | b                             # unión
    return len(a & b) / len(union) if union else 0.0   # el if evita 0/0 con dos vacíos


gustos = ["accion", "rol"]                    # lo que declaró el asistente
catalogo = pd.DataFrame({"stand": ["S1", "S2", "S3"],
                         "tags": [["accion"], ["rol", "accion", "terror"], ["terror"]]})

# map aplica jaccard fila por fila: pandas no sabe de conjuntos, la función sí
catalogo["sim"] = catalogo["tags"].map(lambda t: jaccard(gustos, t))
ver("Stands ordenados por similitud", catalogo.sort_values("sim", ascending=False))

# El desglose, que es lo que hay que saber explicar en la defensa
detalle = catalogo.assign(
    interseccion=catalogo["tags"].map(lambda t: sorted(set(gustos) & set(t))),
    n_inter=catalogo["tags"].map(lambda t: len(set(gustos) & set(t))),
    n_union=catalogo["tags"].map(lambda t: len(set(gustos) | set(t))))
ver("Por qué cada stand obtuvo ese puntaje", detalle[["stand", "interseccion", "n_inter", "n_union", "sim"]])
print("""
  Mira S1 y S2: S1 comparte 1 etiqueta de 2 que tiene el asistente -> 1/2 = 0.50.
  S2 comparte LAS DOS, pero tiene una tercera que el asistente no pidió, así que
  la unión crece a 3 -> 2/3 = 0.67.

  Ahí está el efecto que te va a aparecer en el proyecto: Jaccard CASTIGA a los
  stands con muchas etiquetas. Un stand de cómics con 40 etiquetas tendrá una
  unión enorme y por lo tanto similitud baja, aunque contenga todo lo que al
  asistente le gusta. Ese desbalance entre dominios es una decisión de diseño
  que el grupo tiene que resolver y justificar.""")


▸ Stands ordenados por similitud


,stand,tags,sim
1,S2,"[rol, accion, terror]",0.666667
0,S1,[accion],0.500000
2,S3,[terror],0.000000



▸ Por qué cada stand obtuvo ese puntaje


,stand,interseccion,n_inter,n_union,sim
0,S1,[accion],1,2,0.500000
1,S2,"[accion, rol]",2,3,0.666667
2,S3,[],0,3,0.000000



  Mira S1 y S2: S1 comparte 1 etiqueta de 2 que tiene el asistente -> 1/2 = 0.50.
  S2 comparte LAS DOS, pero tiene una tercera que el asistente no pidió, así que
  la unión crece a 3 -> 2/3 = 0.67.

  Ahí está el efecto que te va a aparecer en el proyecto: Jaccard CASTIGA a los
  stands con muchas etiquetas. Un stand de cómics con 40 etiquetas tendrá una
  unión enorme y por lo tanto similitud baja, aunque contenga todo lo que al
  asistente le gusta. Ese desbalance entre dominios es una decisión de diseño
  que el grupo tiene que resolver y justificar.


---

<a id="sec-11"></a>
## 11. Datos sintéticos: asistentes e interacciones
<sub>[↑ índice](#indice)</sub>

Tus tres fuentes traen el **catálogo**, pero no traen **asistentes** ni
**interacciones**: eso no existe en ningún dataset público y hay que generarlo.

> **NOTA sobre la semilla.** `np.random.default_rng(42)` fija el generador: misma
> semilla → mismos datos → los números de tu informe y los de Cris coinciden. Es
> requisito de reproducibilidad, no una manía.

In [22]:
rng = np.random.default_rng(42)              # generador propio; NO uses np.random.* global

tags_posibles = ["accion", "aventura", "rol", "terror", "ciencia-ficcion"]
stands_ids = [f"stand_{i}" for i in range(5)]

asistentes = pd.DataFrame({
    "_id": [f"asis_{i}" for i in range(4)],
    # Para cada asistente: entre 1 y 3 gustos distintos (replace=False evita repetidos).
    # El tope de 8 que definieron en E1 se aplicaría aquí.
    "intereses_declarados": [
        sorted(rng.choice(tags_posibles, size=int(rng.integers(1, 4)), replace=False).tolist())
        for _ in range(4)],
})
ver("Asistentes sintéticos", asistentes)

interacciones = pd.DataFrame({
    # replace=True (por defecto): un asistente puede interactuar varias veces
    "asistente_id": rng.choice(asistentes["_id"], size=10),
    "objetivo_id":  rng.choice(stands_ids, size=10),
    # p= da las probabilidades: las visitas son comunes, los favoritos raros.
    # Sin p=, los tres tipos saldrían con la misma frecuencia y sería irreal.
    "tipo":         rng.choice(["visita", "asistencia", "favorito"], size=10,
                               p=[0.6, 0.25, 0.15]),
    # Una fecha base + un desfase aleatorio en minutos
    "fecha":        pd.Timestamp("2026-11-01 10:00")
                    + pd.to_timedelta(rng.integers(0, 600, size=10), unit="m"),
})
ver("Interacciones ordenadas por fecha", interacciones.sort_values("fecha"))
ver("¿Se respetaron las proporciones de 'tipo'?",
    interacciones["tipo"].value_counts(normalize=True).round(2))


▸ Asistentes sintéticos


,_id,intereses_declarados
0,asis_0,[terror]
1,asis_1,"[aventura, rol]"
2,asis_2,[terror]
3,asis_3,[accion]



▸ Interacciones ordenadas por fecha


,asistente_id,objetivo_id,tipo,fecha
5,asis_3,stand_3,asistencia,2026-11-01 10:26:00
7,asis_0,stand_4,favorito,2026-11-01 11:32:00
1,asis_3,stand_1,visita,2026-11-01 11:56:00
2,asis_2,stand_0,visita,2026-11-01 13:38:00
3,asis_3,stand_4,asistencia,2026-11-01 14:40:00
4,asis_2,stand_3,asistencia,2026-11-01 14:58:00
6,asis_2,stand_2,visita,2026-11-01 15:27:00
9,asis_1,stand_2,asistencia,2026-11-01 16:49:00



▸ ¿Se respetaron las proporciones de 'tipo'?


tipo
visita        0.4
asistencia    0.4
favorito      0.2
Name: proportion, dtype: float64

In [23]:
ver("sample: filas al azar (random_state lo hace reproducible)",
    catalogo.sample(2, random_state=1))
ver("date_range: horarios de presentaciones cada 2 horas",
    pd.date_range("2026-11-01 10:00", periods=4, freq="2h"))
ver(".dt: extraer partes de una fecha",
    pd.Series(pd.date_range("2026-11-01 10:00", periods=3, freq="h")).dt.hour)

# --- Perfil DEDUCIDO: contador {tag: n} por asistente ---
# Esta es la estructura que definieron en E1 para los intereses deducidos.
fav = pd.DataFrame({"asistente_id": ["a1", "a1", "a2"], "tag": ["accion", "accion", "rol"]})
ver("Eventos crudos (un favorito por fila)", fav)

conteo = fav.groupby(["asistente_id", "tag"]).size()      # Series con índice de DOS niveles
ver("groupby de dos claves -> índice jerárquico", conteo)

# unstack convierte el 2º nivel del índice en columnas; luego cada fila -> dict.
# Importante: se filtran los ceros, si no cada asistente cargaría TODOS los tags.
perfil = (conteo.unstack(fill_value=0)                                   # tags como columnas
                .apply(lambda fila: {k: int(v) for k, v in fila.items() if v > 0}, axis=1))
ver("Perfil deducido {tag: n} por asistente", perfil)
ver("El de a1, en detalle", perfil["a1"])
print("""
  ⚠️ El filtro `if v > 0` no es cosmético. unstack rellena con 0 todas las
  combinaciones asistente×tag que no ocurrieron; sin filtrar, el documento de
  cada asistente llevaría una entrada por CADA etiqueta del sistema, la mayoría
  en cero. Con 300 etiquetas eso son 300 campos inútiles por documento.""")


▸ sample: filas al azar (random_state lo hace reproducible)


,stand,tags,sim
0,S1,[accion],0.5
2,S3,[terror],0.0



▸ date_range: horarios de presentaciones cada 2 horas
DatetimeIndex(['2026-11-01 10:00:00', '2026-11-01 12:00:00', '2026-11-01 14:00:00', '2026-11-01 16:00:00'], dtype='datetime64[us]', freq='2h')

▸ .dt: extraer partes de una fecha


0    10
1    11
2    12
dtype: int32


▸ Eventos crudos (un favorito por fila)


...


▸ groupby de dos claves -> índice jerárquico


asistente_id  tag   
a1            accion    2
a2            rol       1
dtype: int64


▸ Perfil deducido {tag: n} por asistente


asistente_id
a1    {'accion': 2}
a2       {'rol': 1}
dtype: object


▸ El de a1, en detalle
{'accion': 2}

  ⚠️ El filtro `if v > 0` no es cosmético. unstack rellena con 0 todas las
  combinaciones asistente×tag que no ocurrieron; sin filtrar, el documento de
  cada asistente llevaría una entrada por CADA etiqueta del sistema, la mayoría
  en cero. Con 300 etiquetas eso son 300 campos inútiles por documento.


---

<a id="sec-12"></a>
## 12. Salida: guardar y preparar para MongoDB
<sub>[↑ índice](#indice)</sub>

El borde entre pandas y MongoDB es donde fallan todos los ETL la primera vez, por una
razón de fondo: **pandas y BSON tienen sistemas de tipos distintos**. pandas optimiza
para cálculo vectorizado (de ahí `int64`, de ahí `NaN`); BSON optimiza para serializar.

| Tipo de pandas | Problema en BSON | Arreglo |
|---|---|---|
| `np.int64`, `np.float64` | `InvalidDocument: cannot encode object` | `int(v)` / `float(v)`, o `.item()` |
| `NaN` | se inserta **sin error** y contamina las consultas | convertir a `None` |
| `NaT` (fecha nula) | `ValueError: NaTType does not support utcoffset` | convertir a `None` |
| `pd.Timestamp` | se guarda de forma inconsistente | `.to_pydatetime()` |

El segundo es el peligroso: **no falla**. `NaN` es un float válido, así que se inserta
tan contento y después tus consultas devuelven documentos con `nan` dentro.

In [24]:
OUT = Path("datos") / "interim"          # carpeta de trabajo del proyecto
print("Los intermedios irían a:", OUT)

# Formatos de salida y cuándo usar cada uno:
#   to_csv     -> universal y legible, pero PIERDE los tipos (todo vuelve como texto)
#   to_parquet -> conserva tipos y pesa mucho menos; requiere pyarrow
#   to_json    -> con lines=True queda igual que la entrada de Goodreads
buffer = StringIO()
stands.to_csv(buffer, index=False)                # index=False: no escribir 0,1,2...
buffer.seek(0)
de_vuelta = pd.read_csv(buffer)
ver("Guardado y releído desde CSV", de_vuelta)
ver("⚠️ La lista de tags volvió como TEXTO, no como lista",
    {"tipo original": type(stands["tags"].iloc[0]).__name__,
     "tipo tras CSV": type(de_vuelta["tags"].iloc[0]).__name__})
print("""
  Ese es el argumento para usar Parquet (o JSON Lines) en los intermedios: CSV
  convierte ['accion','rol'] en el string "['accion', 'rol']", y tendrías que
  volver a parsearlo con ast.literal_eval en cada paso del pipeline.""")

Los intermedios irían a: datos/interim

▸ Guardado y releído desde CSV


,Publisher,n_titulos,tags
0,acme,2,"['accion', 'rol-fantasia']"
1,omega,1,['rol-fantasia']
2,zeta co,1,"['accion', 'ciencia-ficcion']"



▸ ⚠️ La lista de tags volvió como TEXTO, no como lista
{'tipo original': 'list', 'tipo tras CSV': 'str'}

  Ese es el argumento para usar Parquet (o JSON Lines) en los intermedios: CSV
  convierte ['accion','rol'] en el string "['accion', 'rol']", y tendrías que
  volver a parsearlo con ast.literal_eval en cada paso del pipeline.


In [25]:
def a_documentos(df: pd.DataFrame) -> list[dict]:
    """DataFrame -> lista de dicts seguros para BSON.

    Hace las cuatro conversiones de la tabla de arriba de una vez. Es barata y
    evita el 100 % de los InvalidDocument: úsala SIEMPRE como última escala
    antes de insert_many.
    """
    limpio = df.copy()
    for col in limpio.columns:
        s = limpio[col]
        if pd.api.types.is_datetime64_any_dtype(s):
            # dtype=object es obligatorio: si no, pandas vuelve a inferir
            # datetime64 y reinyecta el NaT que acabamos de quitar.
            limpio[col] = pd.Series(
                [None if pd.isna(v) else v.to_pydatetime() for v in s],
                index=limpio.index, dtype=object)
        elif pd.api.types.is_integer_dtype(s):
            limpio[col] = s.astype(object).map(lambda v: None if pd.isna(v) else int(v))
        elif pd.api.types.is_float_dtype(s):
            limpio[col] = s.astype(object).map(lambda v: None if pd.isna(v) else float(v))
    # Última pasada: los NaN sueltos en columnas de tipo object
    return [{k: (None if (isinstance(v, float) and np.isnan(v)) else v)
             for k, v in registro.items()}
            for registro in limpio.to_dict("records")]


prueba = pd.DataFrame({"n": pd.array([1, 2, None], dtype="Int64"),
                       "precio": [9.9, np.nan, 3.0],
                       "fecha": pd.to_datetime(["2026-11-02", "2026-11-16", None])})
ver("DataFrame con los tres tipos problemáticos", prueba)

docs = a_documentos(prueba)
ver("Tras a_documentos: tipos de Python puros",
    [{k: (type(v).__name__, v) for k, v in d.items()} for d in docs])

# La verificación que de verdad cuenta: ¿lo acepta BSON?
try:
    from bson import BSON
    for d in docs:
        BSON.encode(d)
    ver("¿BSON los acepta?", "✅ sí, los 3 documentos son válidos")
except ImportError:
    ver("bson no instalado aquí", "la verificación corre en NB-02")

ver("Los stands, listos para insert_many", a_documentos(stands)[:2])
print("""
🎯 Para tu proyecto
   # from pymongo import MongoClient
   # cli = MongoClient("mongodb://localhost:27017")
   # db = cli["gamecon"]
   # db.stands.insert_many(a_documentos(stands))     # por lotes si son muchos
   # db.stands.create_index("tags")                  # multikey: una entrada POR etiqueta

   Decisión que el grupo tiene que tomar: ¿qué índices crean? La respuesta sale
   de sus 4 consultas principales, no al revés. Cada índice acelera lecturas y
   encarece escrituras; crearlos "por si acaso" es un costo sin beneficio.""")


▸ DataFrame con los tres tipos problemáticos


,n,precio,fecha
0,1,9.9,2026-11-02
1,2,NaN,2026-11-16
2,<NA>,3.0,NaT



▸ Tras a_documentos: tipos de Python puros
[{'n': ('float', 1.0), 'precio': ('float', 9.9), 'fecha': ('datetime', datetime.datetime(2026, 11, 2, 0, 0))}, {'n': ('float', 2.0), 'precio': ('NoneType', None), 'fecha': ('datetime', datetime.datetime(2026, 11, 16, 0, 0))}, {'n': ('NoneType', None), 'precio': ('float', 3.0), 'fecha': ('NoneType', None)}]

▸ ¿BSON los acepta?
✅ sí, los 3 documentos son válidos

▸ Los stands, listos para insert_many
[{'Publisher': 'acme', 'n_titulos': 2, 'tags': ['accion', 'rol-fantasia']}, {'Publisher': 'omega', 'n_titulos': 1, 'tags': ['rol-fantasia']}]

🎯 Para tu proyecto
   # from pymongo import MongoClient
   # cli = MongoClient("mongodb://localhost:27017")
   # db = cli["gamecon"]
   # db.stands.insert_many(a_documentos(stands))     # por lotes si son muchos
   # db.stands.create_index("tags")                  # multikey: una entrada POR etiqueta

   Decisión que el grupo tiene que tomar: ¿qué índices crean? La respuesta sale
   de sus 4 consultas princ

---

<a id="sec-13"></a>
## 13. Cheat sheet y mapa del proyecto
<sub>[↑ índice](#indice)</sub>

### Qué función va en qué paso

| Paso del ETL | Funciones |
|---|---|
| Cargar las 3 fuentes | `read_csv`, `read_json(lines=True)`, `usecols`, `nrows`, `chunksize` |
| Perfilar | `shape`, `dtypes`, `info`, `describe`, `isna().sum()`, `duplicated`, `nunique`, `value_counts` |
| Quedarse con lo útil | `[[cols]]`, `drop`, `rename`, `dropna(subset=)`, `query`, `isin` |
| Normalizar géneros | `.str.strip/lower/replace`, `unicodedata` + `map`, `split`, `explode` |
| Tabla de mapeo | `dict` + `map`, `merge(how='left')`, `isna()` para ver faltantes |
| Stands por productor | `groupby().agg(..., lambda s: sorted(set(s)))`, `assign`, `concat` |
| Colección `tags` | `value_counts`, `groupby('tag')['origen'].agg(list)`, `np.log` (IDF) |
| Presentaciones | `date_range`, `to_datetime`, `sample`, `assign` |
| Asistentes / interacciones | `default_rng`, `choice`, `to_timedelta` |
| Perfil deducido | `groupby([asis, tag]).size()`, `unstack`, `to_dict` |
| Exportar / cargar | `to_parquet`, `a_documentos()`, `insert_many` |

### Las distinciones que se confunden

| | |
|---|---|
| `.loc` vs `.iloc` | etiqueta (fin incluido) vs posición (fin excluido) |
| `map` vs `replace` | lo no encontrado → `NaN` vs se queda igual |
| `agg` vs `transform` | una fila por **grupo** vs una fila por **fila** |
| `$in` mental: `isin` vs `all` | "al menos uno" vs "todos" |
| `int64` vs `Int64` | no admite nulos vs sí los admite |
| `merge` vs `concat` | unir por clave vs apilar |

### Plantilla mínima de un paso del ETL

```python
def transformar(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)                                   # 1. medir antes
    df = (df.copy()                                # 2. nunca mutar la entrada
            .dropna(subset=["productor"])
            .assign(tag=lambda d: normalizar(d["genero"])))
    print(f"{n0} -> {len(df)} filas ({n0 - len(df)} descartadas)")   # 3. reportar siempre
    return df
```

Las tres líneas marcadas son el patrón: **medir, no mutar, reportar**. Un paso del ETL
que no dice cuántas filas descartó no se puede auditar.

---

<a id="sec-14"></a>
## 14. Conexiones con el mundo real
<sub>[↑ índice](#indice)</sub>

**Big Data (CIT-2309).** Esta guía es la referencia directa de la Entrega 2. El ETL que
entreguen es exactamente: cargar → perfilar → normalizar → mapear → agrupar → documentos.

**La industria.** El 80 % del tiempo de un proyecto de datos se va en lo que cubre esta
guía, no en el modelo. Es el consenso de todos los libros de la biblioteca
(Géron lo dice en el cap. 2, Kleppmann lo da por supuesto).

**Escala.** Con millones de filas, `groupby` + `apply` en Python se vuelve inviable y se
pasa a Spark o a Polars. Pero **las operaciones son las mismas**: lo que aprendes aquí
se traduce casi uno a uno. Eso es NB-22.

**Tesis.** Un registro de experimentos (parámetros de la SNN + métricas de ojo + BER) es
una tabla que se perfila, se limpia y se guarda igual que esto.

---

<a id="sec-15"></a>
## 15. Ejercicios progresivos
<sub>[↑ índice](#indice)</sub>

### Nivel 1 — Reproducir (guiado): perfil completo de una fuente

Escribe `resumen_fuente(df, col_productor, col_genero)` que devuelva, en un solo dict:
número de filas, productores distintos, % de nulos en productor, mediana y máximo de
géneros por fila, y los 5 productores más frecuentes. Pruébalo con `toy`.

In [26]:
# Tu turno (Nivel 1)

#### ✅ Solución — Nivel 1

In [27]:
def resumen_fuente(df: pd.DataFrame, col_productor: str, col_genero: str,
                   sep: str = ",") -> dict:
    """Las seis cifras que deciden si una fuente sirve para el proyecto."""
    # Géneros por fila: split -> longitud de la lista. Los nulos cuentan como 0.
    n_gen = (df[col_genero].fillna("").str.split(sep)
               .map(lambda xs: len([x for x in xs if x.strip()])))
    return {
        "filas": len(df),
        "productores_distintos": int(df[col_productor].nunique()),
        "nulos_productor_%": round(df[col_productor].isna().mean() * 100, 1),
        "generos_por_fila_mediana": float(n_gen.median()),
        "generos_por_fila_max": int(n_gen.max()),
        "top5_productores": df[col_productor].value_counts().head(5).to_dict(),
    }

for k, v in resumen_fuente(toy, "Publisher", "Genres").items():
    print(f"  {k:28} {v}")
print("""
  La cifra decisiva es 'generos_por_fila_mediana'. Con mediana 1, Jaccard entre
  dos ítems solo puede dar 0 o 1: deja de medir parecido y pasa a medir
  igualdad. Si una de tus fuentes da 1, hay que combinar columnas (como
  Genres + Themes en el dataset de anime) o cambiar de medida.""")

  filas                        7
  productores_distintos        4
  nulos_productor_%            14.3
  generos_por_fila_mediana     1.0
  generos_por_fila_max         3
  top5_productores             {'Zeta Co': 2, 'Omega': 2, ' Acme ': 1, 'acme': 1}

  La cifra decisiva es 'generos_por_fila_mediana'. Con mediana 1, Jaccard entre
  dos ítems solo puede dar 0 o 1: deja de medir parecido y pasa a medir
  igualdad. Si una de tus fuentes da 1, hay que combinar columnas (como
  Genres + Themes en el dataset de anime) o cambiar de medida.


### Nivel 2 — Aplicar: detectar si un dataset es sintético

Los datos reales siguen una **ley de potencias**: unos pocos productores concentran la
mayoría de los títulos. Los datasets inventados tienden a repartir parejo.

Escribe `parece_sintetico(serie)` que calcule qué fracción de los registros concentra el
10 % de productores más frecuentes, y lo compare con lo esperado en una distribución
uniforme. Pruébalo con una serie real (sesgada) y una uniforme.

In [28]:
# Tu turno (Nivel 2)

#### ✅ Solución — Nivel 2

In [29]:
def parece_sintetico(serie: pd.Series, umbral: float = 0.35) -> dict:
    """Mide la concentración del top 10 % de valores.

    En una distribución UNIFORME el 10 % de los valores concentra ~10 % de los
    registros. En datos reales con ley de potencias concentra mucho más (50-80 %).
    """
    conteo = serie.value_counts()                       # frecuencia de cada valor, ordenada
    k = max(1, int(len(conteo) * 0.10))                 # cuántos valores son el 10 % superior
    concentracion = conteo.head(k).sum() / conteo.sum() # qué fracción de registros acaparan
    return {"valores_distintos": len(conteo), "top10%_son": k,
            "concentran": f"{concentracion:.1%}", "esperado_si_uniforme": "≈10%",
            "veredicto": "parece SINTÉTICO" if concentracion < umbral else "parece REAL"}

rg = np.random.default_rng(0)
# Zipf genera exactamente una ley de potencias: así se ven los datos reales
real = pd.Series(rg.zipf(1.6, 5000)).clip(upper=400)
uniforme = pd.Series(rg.integers(0, 400, 5000))

for nombre, s in [("ley de potencias (realista)", real), ("uniforme (sospechoso)", uniforme)]:
    print(f"\n  {nombre}")
    for k, v in parece_sintetico(s).items():
        print(f"     {k:22} {v}")
print("""
  Esta es la prueba que te pedí hacer con el 'Comic Books Dataset (10,000
  Entries)'. Si sus editoriales reparten parejo, el dataset está generado y no
  sirve como evidencia de un catálogo real — aunque los datos "se vean bien".""")


  ley de potencias (realista)
     valores_distintos      196
     top10%_son             19
     concentran             89.1%
     esperado_si_uniforme   ≈10%
     veredicto              parece REAL

  uniforme (sospechoso)
     valores_distintos      400
     top10%_son             40
     concentran             15.0%
     esperado_si_uniforme   ≈10%
     veredicto              parece SINTÉTICO

  Esta es la prueba que te pedí hacer con el 'Comic Books Dataset (10,000
  Entries)'. Si sus editoriales reparten parejo, el dataset está generado y no
  sirve como evidencia de un catálogo real — aunque los datos "se vean bien".


### Nivel 3 — Integrar (con NB-02 y NB-03): el ETL completo de una fuente

Encadena todo lo de esta guía en **una sola función** `procesar_fuente()` que reciba un
DataFrame crudo y devuelva `(stands, informe)`, donde `informe` cuenta cuántas filas se
descartaron y por qué. La ecuación `leídos = cargados + rechazados` debe cuadrar.

Es el patrón de NB-02 §17 y es lo que se evalúa en la Entrega 2.

In [30]:
# Tu turno (Nivel 3)

#### ✅ Solución — Nivel 3

In [31]:
from collections import Counter

def procesar_fuente(crudo: pd.DataFrame, dominio: str, mapa: dict,
                    col_prod="Publisher", col_gen="Genres", col_id="Name"):
    """Crudo -> (stands, informe). Nunca descarta nada en silencio."""
    informe = Counter()
    informe["leidos"] = len(crudo)
    df = crudo.copy()

    # R1 · sin productor -> no puede ser un stand
    sin_prod = df[col_prod].isna()
    informe["rechazo_sin_productor"] = int(sin_prod.sum())
    df = df[~sin_prod]

    # R2 · duplicados exactos
    dups = df.duplicated().sum()
    informe["rechazo_duplicado"] = int(dups)
    df = df.drop_duplicates()

    # R3 · normalizar el productor ANTES de agrupar (si no, "Acme" y "acme" son 2 stands)
    df[col_prod] = normalizar(df[col_prod])

    # R4 · géneros -> forma larga -> mapear al vocabulario común
    df["_tags"] = df[col_gen].fillna("").str.split(",")
    largo_f = df.explode("_tags")
    largo_f["_tags"] = normalizar(largo_f["_tags"])
    largo_f["tag"] = largo_f["_tags"].map(mapa)

    desconocidas = largo_f.loc[largo_f["tag"].isna() & (largo_f["_tags"] != ""), "_tags"]
    informe["etiquetas_desconocidas"] = int(desconocidas.nunique())

    # R5 · un ítem sin ninguna etiqueta válida no sirve para recomendar
    validos = largo_f.dropna(subset=["tag"])
    ids_con_tag = set(validos[col_id])
    sin_tag = df[~df[col_id].isin(ids_con_tag)]
    informe["rechazo_sin_etiquetas"] = int(len(sin_tag))

    stands_f = (validos.groupby(col_prod)
                       .agg(n_titulos=(col_id, "nunique"),
                            tags=("tag", lambda s: sorted(set(s))))
                       .reset_index()
                       .assign(dominio=dominio))
    informe["cargados"] = int(validos[col_id].nunique())
    return stands_f, informe, sorted(desconocidas.unique())


mapa_demo = dict(zip(mapeo["origen"], mapeo["tag"]))
st, inf, desc = procesar_fuente(toy, "juego", mapa_demo)
ver("Stands producidos", st)
ver("Informe del ETL", dict(inf))
ver("Etiquetas que faltan en la tabla de mapeo", desc)

cuadra = inf["leidos"] == (inf["cargados"] + inf["rechazo_sin_productor"]
                           + inf["rechazo_duplicado"] + inf["rechazo_sin_etiquetas"])
ver("¿Cuadra leídos = cargados + rechazados?", cuadra)
print("""
  Esa última comprobación es la que tienes que poder mostrar en la defensa. Un
  ETL que dice "cargué 14.300 registros" sin decir cuántos botó y por qué no se
  puede auditar, y es lo primero que te van a preguntar.

  Si te da False, el error típico es contar dos veces una fila que cae en dos
  reglas. Arreglarlo obliga a decidir el ORDEN de las reglas, que también es una
  decisión de diseño.""")


▸ Stands producidos


,Publisher,n_titulos,tags,dominio
0,acme,2,"[accion, rol-fantasia]",juego
1,zeta-co,1,"[accion, ciencia-ficcion]",juego



▸ Informe del ETL
{'leidos': 7, 'rechazo_sin_productor': 1, 'rechazo_duplicado': 1, 'etiquetas_desconocidas': 1, 'rechazo_sin_etiquetas': 2, 'cargados': 3}

▸ Etiquetas que faltan en la tabla de mapeo
['drama']

▸ ¿Cuadra leídos = cargados + rechazados?
True

  Esa última comprobación es la que tienes que poder mostrar en la defensa. Un
  ETL que dice "cargué 14.300 registros" sin decir cuántos botó y por qué no se
  puede auditar, y es lo primero que te van a preguntar.

  Si te da False, el error típico es contar dos veces una fila que cae en dos
  reglas. Arreglarlo obliga a decidir el ORDEN de las reglas, que también es una
  decisión de diseño.


---

<a id="sec-16"></a>
## 16. Glosario de errores frecuentes
<sub>[↑ índice](#indice)</sub>

| Error | Causa | Arreglo |
|---|---|---|
| `SettingWithCopyWarning` | modificar una vista de otro DataFrame | `.copy()` después de filtrar |
| `x == np.nan` siempre `False` | `NaN` no es igual ni a sí mismo | `isna()` / `notna()` |
| `Cannot mask with non-boolean array containing NA` | `.str.contains` con nulos | agregar `na=False` |
| `ValueError` al combinar condiciones | falta de paréntesis | `(a > 1) & (b < 2)` |
| El `merge` multiplicó filas | clave repetida en la tabla derecha | `validate="one_to_one"`, comparar `len()` |
| Años impresos como `2001.0` | `int64` no admite nulos y pasa a float | `astype("Int64")` |
| `InvalidDocument: cannot encode object` | `np.int64` no es BSON | `a_documentos()` |
| Documentos con `nan` dentro | `NaN` **se inserta sin error** | convertir a `None` antes |
| `"Acme"` y `"acme"` son dos stands | no se normalizó antes de agrupar | `normalizar()` antes de `groupby` |
| `KeyError` tras un `groupby` | la clave quedó como índice | `.reset_index()` |

---

<a id="sec-17"></a>
## 17. Resumen y siguiente paso
<sub>[↑ índice](#indice)</sub>

### Resumen en 5 líneas

1. **Perfila antes de limpiar**, y limpia según el modelo destino — no al revés.
2. **Normalizar texto es la decisión que hace o rompe** el cruce entre dominios.
3. `explode` ↔ `groupby(...).agg(list)` son el ida y vuelta entre forma larga y ancha.
4. La **tabla de mapeo es un artefacto humano**: un CSV que se edita y se muestra.
5. El borde pandas → BSON necesita **conversión explícita de tipos**; `NaN` es el
   peligroso porque no falla.

### Siguiente paso

→ **NB-02 — Pandas + MongoDB para Big Data**: el flujo completo con un caso de ejemplo
de punta a punta, incluyendo índices, pipelines de agregación y el ETL idempotente.
Esta guía es el catálogo; NB-02 es el recorrido.

### Para profundizar

- VanderPlas, *Python Data Science Handbook*, cap. 13–24.
- McKinney, *Python for Data Analysis*, cap. 5–8 — el catálogo de referencia por
  excelencia, con el formato `In [ ]:` / `Out[ ]:` para cada operación.